In [129]:
! pip install perplexityai openai requests

In [130]:
import os

PERPLEXITY_API_KEY = os.environ["PERPLEXITY_API_KEY"]
MORPH_API_KEY = os.environ["MORPH_API_KEY"]
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

In [131]:
from perplexity import Perplexity

client = Perplexity(
    api_key=PERPLEXITY_API_KEY
)

SEARCH_STRING = "latest AI developments 2024"

search = client.search.create(
    query=SEARCH_STRING,
    max_results=1,
    max_tokens_per_page=4096
)

for result in search.results:
    print(f"{result.title}: {result.url}")

Embodied Ai And World Models: https://www.ibm.com/think/insights/artificial-intelligence-trends


In [132]:
import shutil
import subprocess

CURL = shutil.which("curl") or "curl"
MAX_BODY_CHARS_PER_URL = 1_000_000


def curl_get(url: str, timeout: int = 90) -> tuple[int, str, str]:
    """Run a real `curl` GET and return (exit_code, stdout, stderr)."""
    cmd = [
        CURL,
        "-sS",
        "-L",
        "-g",
        "-X",
        "GET",
        "-A",
        "Mozilla/5.0 (compatible; perplexity-notebook/1.0)",
        "--max-time",
        str(timeout),
        url,
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    err = proc.stderr if isinstance(proc.stderr, str) else (proc.stderr or b"").decode(
        "utf-8", errors="replace"
    )
    return proc.returncode, proc.stdout, err


parts: list[str] = []
for i, result in enumerate(search.results, start=1):
    code, body, err = curl_get(result.url)
    if code != 0:
        body = f"[curl GET failed exit={code} stderr={err.strip()}]"
    elif len(body) > MAX_BODY_CHARS_PER_URL:
        body = body[:MAX_BODY_CHARS_PER_URL] + "\n... [truncated]"

    parts.append(
        f"=== Source {i} ===\n"
        f"Title: {result.title}\n"
        f"URL: {result.url}\n\n"
        f"{body}\n"
    )

web_search_content = "\n\n".join(parts)
print(
    f"web_search_content: {len(web_search_content)} chars from {len(search.results)} curl GETs"
)

web_search_content: 201053 chars from 1 curl GETs


In [ ]:
import requests

# String input
response = requests.post(
    "https://api.morphllm.com/v1/compact",
    headers={"Authorization": "Bearer " + MORPH_API_KEY},
    json={
        "input": web_search_content,
        "query": SEARCH_STRING,
        "compression_ratio": 0.2,
        "preserve_recent": 0,
    },
)

data = response.json()
compacted_web_search_content = data["output"]
print(len(compacted_web_search_content))

117139


In [134]:
import openai

client = openai.OpenAI(
    api_key=OPENAI_API_KEY,
)

In [135]:
# question generation
response = client.responses.create(
    model="gpt-5.4-nano-2026-03-17",
    reasoning={"effort": "low"},
    input=[
        {"role": "user", "content": SEARCH_STRING + " Based on this general question, and the websearch contents i showed you, output exactly 10 sub questions that are all answerable by reading websearch contents. Exactly one question per line."},
        {"role": "user", "content": web_search_content},
    ],
)

questions = response.output_text
print(questions)


What does the IBM article say caused a dramatic decrease in inference costs, and how is that illustrated (e.g., the SemiAnalysis chart and the GPT-4 vs IBM Granite comparison)?  
What are “reasoning models” in the article, and how does the piece explain that test-time compute (inference scaling) can improve performance?  
According to the article, what problem do reasoning models create in practice (inference costs, latency, and impact on context windows), and what solution does it propose (“hybrid reasoning models”)?  
What “hybrid reasoning” capabilities are described in the article for IBM Granite (thinking mode toggle) and other providers (e.g., Anthropic Claude 3.7 Sonnet, Google Gemini 2.5 Flash, Alibaba Qwen3)?  
What issue does the article raise as a “drain on digital resources,” including specific examples like Wikimedia traffic growth and the “DDoS on the entire internet” quote?  
What defensive measures and initiatives does the article mention against AI crawlers/bot scrapin

In [136]:
# WITHOUT COMPACTION

response = client.responses.create(
    model="gpt-5.4-nano-2026-03-17",
    reasoning={"effort": "low"},
    input=[
        {"role": "system", "content": "You are a helpful assistant. The user will ask you 10 questions and provide websearch content. Your must answer each question, in order in exactly 3 sentences."},
        {"role": "user", "content": questions},
        {"role": "user", "content": web_search_content},
    ],
)


answers_uncompacted = response.output_text
print(answers_uncompacted)

1) The IBM article says the dramatic decrease in inference costs was driven by rapid improvements in model efficiency and algorithmic progress—“per-token pricing … decreased dozens of times” for equivalent MMLU results, illustrated by the SemiAnalysis per-token chart. It also illustrates the trend by comparing the original GPT-4’s reported HumanEval score (about 67%) with a much smaller IBM Granite 3.3 2B Instruct model achieving about 80.5%. The overall point is that models reached similar or better outcomes with far less compute, enabling cheaper inference at scale.  

2) In the article, “reasoning models” are LLMs trained/optimized to perform better on tasks that require logical decision-making by scaling *test-time compute* (inference scaling). The piece explains that spending more inference-time computation—i.e., generating additional “thinking” tokens before the final answer—can improve performance, because research shows that increasing test-time compute can help as much as incr

In [137]:
#WITH COMPACTION

response = client.responses.create(
    model="gpt-5.4-nano-2026-03-17",
    reasoning={"effort": "low"},
    input=[
        {"role": "system", "content": "You are a helpful assistant. The user will ask you 10 questions and provide websearch content. Your must answer each question, in order in exactly 3 sentences."},
        {"role": "user", "content": questions},
        {"role": "user", "content": compacted_web_search_content},
    ],
)


answers_compacted = response.output_text
print(answers_compacted)

1) The IBM article says the dramatic decrease in inference costs was caused by accelerating improvements that let similar results be achieved with far less compute, as illustrated by SemiAnalysis’ per-token pricing chart over time and by the example that IBM Granite 3.3 2B Instruct (900× smaller) reached 80.5% vs GPT-4’s ~67% HumanEval despite being much smaller. It frames this as aggregated “model economy” improving faster than most people realize when looking across multiple releases. The chart shows per-token cost dropping “dozens of times” within under two years, while the GPT-4 vs Granite comparison makes the same point with a concrete performance/size contrast.

2) In the article, “reasoning models” are models tuned to perform better on tasks that require logical decision-making, often by doing extra computation at generation time. The piece explains that test-time compute (inference scaling)—i.e., allowing the model to “think” longer using more compute—can improve performance up

In [138]:
import re


def split_numbered_blocks(text: str) -> list[str]:
    """Split top-level numbered answers like '1. ...' or '1) ...'.

    We only split on true item starts at the beginning of a line. This avoids
    over-splitting on inline references like '(1)' inside an answer body.
    """
    text = text.strip()
    if not text:
        return []

    pattern = r"^\s*(?:[1-9]|10)[.)]\s"
    starts = [m.start() for m in re.finditer(pattern, text, flags=re.MULTILINE)]

    if len(starts) < 2:
        return [text]

    blocks: list[str] = []
    for i, start in enumerate(starts):
        end = starts[i + 1] if i + 1 < len(starts) else len(text)
        blocks.append(text[start:end].strip())
    return blocks


question_blocks = [line.strip() for line in questions.splitlines() if line.strip()]
uncomp_blocks = split_numbered_blocks(answers_uncompacted)
comp_blocks = split_numbered_blocks(answers_compacted)

if not (len(question_blocks) == len(uncomp_blocks) == len(comp_blocks) == 10):
    raise ValueError(
        "Expected 10 questions and 10 answer pairs; got "
        f"questions={len(question_blocks)} uncompacted={len(uncomp_blocks)} compacted={len(comp_blocks)}"
    )

JUDGE_MODEL = "gpt-5.4-nano-2026-03-17"

judge_verdicts: list[bool] = []

for idx in range(10):
    sub_q = question_blocks[idx]
    ans_a = uncomp_blocks[idx]
    ans_b = comp_blocks[idx]

    judge_resp = client.responses.create(
        model=JUDGE_MODEL,
        reasoning={"effort": "low"},
        input=[
            {
                "role": "system",
                "content": (
                    "You are a strict judge. You receive the full uncompacted web source, "
                    "one sub-question, and two answers (Answer A = from full web text, "
                    "Answer B = from compacted web text). Decide whether both answers are "
                    "talking about the same substantive thing relative to the source. "
                    "Minor wording differences are OK. Reply false if they disagree materially, "
                    "omit the core point, or are off-topic. Reply with exactly one word: true or false."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"--- SUB-QUESTION ---\n{sub_q}\n\n"
                    f"--- ANSWER A ---\n{ans_a}\n\n"
                    f"--- ANSWER B ---\n{ans_b}\n\n"
                    f"--- UNCOMPACTED WEB SOURCE ---\n{web_search_content}"
                ),
            },
        ],
    )

    raw = judge_resp.output_text.strip().lower()
    same = raw == "true" or raw.startswith("true")
    judge_verdicts.append(same)
    print(f"{idx + 1}/10 judge: {same} (model said: {raw!r})")

print(f"same_topic_count: {sum(judge_verdicts)} / 10")

1/10 judge: True (model said: 'true')
2/10 judge: True (model said: 'true')
3/10 judge: True (model said: 'true')
4/10 judge: True (model said: 'true')
5/10 judge: True (model said: 'true')
6/10 judge: True (model said: 'true')
7/10 judge: True (model said: 'true')
8/10 judge: True (model said: 'true')
9/10 judge: True (model said: 'true')
10/10 judge: True (model said: 'true')
same_topic_count: 10 / 10
